In [17]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from pycds import Variable
from crmprtd.infer import infer
import os
import pickle
import logging
from crmprtd.more_itertools import tap, log_progress
from crmprtd.align import align
from crmprtd.insert import insert
from pprint import pprint
from crmprtd import Row

log = logging.getLogger(__name__)

engine = sa.create_engine("postgresql://tongli1997@dbtest04.pcic.uvic.ca:5432/crmp", echo=False)
session = Session(engine)

# result = session.execute(sa.text("SELECT * FROM crmp.meta_network"))
# test = result.fetchall()

# print(test)

results2 = session.query(Variable).filter(Variable.network_id == 11)
test2 = results2.all()
pprint(test2)

names = [v.name for v in test2]
print(names)
print(len(names))


[<Variable id=534 name='RHx' standard_name='relative_humidity' cell_method='time: maximum' network_id=11>,
 <Variable id=506 name='WindSpeedms' standard_name='wind_speed' cell_method='time: mean' network_id=11>,
 <Variable id=538 name='Wind_n' standard_name='wind_speed' cell_method='time: minimum' network_id=11>,
 <Variable id=504 name='RH' standard_name='relative_humidity' cell_method='time: mean' network_id=11>,
 <Variable id=505 name='DewPtC' standard_name='dew_point_temperature' cell_method='time: mean' network_id=11>,
 <Variable id=502 name='Pressurembar' standard_name='air_pressure' cell_method='time: point' network_id=11>,
 <Variable id=501 name='Rainmm' standard_name='lwe_thickness_of_precipitation_amount' cell_method='time: sum' network_id=11>,
 <Variable id=529 name='Tm' standard_name='air_temperature' cell_method='time: mean' network_id=11>,
 <Variable id=530 name='Tx' standard_name='air_temperature' cell_method='time: maximum' network_id=11>,
 <Variable id=531 name='Tn' sta

In [12]:
import os
print(os.environ.get("HOME"))

/home/vscode


In [13]:
# pprint(test2[0].__dict__)
# pprint(vars(test2[0]))
# pprint(test2[0].__table__.columns.keys())

# Pretty view of all attribute names and values
for key, val in vars(test2[0]).items():
    if not key.startswith('_'):  # skip internal SQLAlchemy attributes
        print(f"{key}: {val}")

id: 534
cell_method: time: maximum
precision: None
description: None
unit: %
display_name: Relative Humidity (Max.)
short_name: relative_humidity_maximum
network_id: 11
mod_time: 2025-02-11 16:03:39.747374
mod_user: crmp
name: RHx
standard_name: relative_humidity


In [14]:
output_folder = '/workspaces/crmprtd/rows_output/'

file_name = 'rows_output_AtlSch.pickle'
file_name = 'rows_output_Barren.pickle'

file_path = os.path.join(output_folder, file_name)

# Load the pickle file
with open(file_path, 'rb') as f:
    rows = pickle.load(f)

# pprint(rows)

variable_names = [r.variable_name for r in rows]
print(variable_names) 
print(len(variable_names)) 

['Rain', 'Rain', 'Rain', 'Rain', 'Rain', 'Rain', 'Rain', 'Rain', 'Rain', 'Rain', 'Pressure', 'Pressure', 'Pressure', 'Pressure', 'Pressure', 'Pressure', 'Pressure', 'Pressure', 'Pressure', 'Pressure', 'Temp', 'Temp', 'Temp', 'Temp', 'Temp', 'Temp', 'Temp', 'Temp', 'Temp', 'Temp', 'RH', 'RH', 'RH', 'RH', 'RH', 'RH', 'RH', 'RH', 'RH', 'RH', 'DewPt', 'DewPt', 'DewPt', 'DewPt', 'DewPt', 'DewPt', 'DewPt', 'DewPt', 'DewPt', 'DewPt', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Wind Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Gust Speed', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Wind Direction', 'Solar Radiation', 'Solar Radiation', 'Solar Radiation', 'Solar Radiation', 'Solar Radiation', 'Solar Radiati

In [15]:
from rapidfuzz import process, fuzz

# names from SQL
db_names = [v.name for v in test2]

# variable_names from your Row objects
local_names = [r.variable_name for r in rows]

# create a mapping: local name -> best database name
mapping = {}
for local_name in local_names:
    match, score, _ = process.extractOne(local_name, db_names, scorer=fuzz.ratio)
    mapping[local_name] = (match, score)

print("Matching results:")
for k, (m, s) in mapping.items():
    print(f"{k:20s} -> {m:20s} (score={s:.1f})")

Matching results:
Rain                 -> Rainmm               (score=80.0)
Pressure             -> Pressurembar         (score=80.0)
Temp                 -> TempC                (score=88.9)
RH                   -> RH                   (score=100.0)
DewPt                -> DewPtC               (score=90.9)
Wind Speed           -> WindSpeedms          (score=85.7)
Gust Speed           -> GustSpeedms          (score=85.7)
Wind Direction       -> WindDirection        (score=96.3)
Solar Radiation      -> SolarRadiationWm     (score=90.3)
Wetness              -> Wetness              (score=100.0)
Snow depth raw       -> Snow_Depth           (score=66.7)
Snow depth           -> Snow_Depth           (score=80.0)
Water Content        -> Water_Content_m3_m3_5_cm (score=64.9)
WC_cal               -> Wind_n               (score=33.3)
Soil Temp            -> Soil_Temp            (score=88.9)
Temp 2               -> TempC                (score=72.7)
RH 2                 -> RH                   (sc

['RHx', 'WindSpeedms', 'Wind_n', 'RH', 'DewPtC', 'Pressurembar', 'Rainmm', 'Tm', 'Tx', 'Tn', 'TempC', 'WindDirection', 'RHn', 'GustSpeedms', 'Tx_Climatology', 'SolarRadiationWm', 'Wetness', 'Tn_Climatology', 'T_mean_Climatology', 'Precip_Climatology', 'Soil_Temp', 'Water_Content_m3_m3_15_cm', 'Water_Content_m3_m3_30_cm', 'Water_Content_m3_m3_5_cm', 'Snow_Depth', 'Wetness_20cm']

Wrong cases:

Snow depth raw       -> Snow_Depth           (score=66.7)  what the raw means?
Water Content        -> Water_Content_m3_m3_5_cm (score=64.9) which depth?
WC_cal               -> Wind_n               (score=33.3). calibrated soil water content?, cubic meter of water per cubic meter of soil
Temp 2               -> TempC                (score=72.7) no match in db
RH 2                 -> RH                   (score=66.7) no match in db
DewPt 2              -> DewPtC               (score=76.9) no match in db

In [28]:
# list of local variable names that should NOT be updated
exclude_vars = [
    "Snow depth raw",
    "Water Content",
    "WC_cal",
    "Temp 2",
    "RH 2",
    "DewPt 2"
]

import logging

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

exclude_vars = [
    "Snow depth raw",
    "Water Content",
    "WC_cal",
    "Temp 2",
    "RH 2",
    "DewPt 2"
]

updated_rows = []
updated_vars = []
unchanged_vars = []

for r in rows:
    best_match, _ = mapping[r.variable_name]
    
    if r.variable_name not in exclude_vars:
        new_var = best_match
        updated_vars.append((r.variable_name, new_var))
        log.info(f"UPDATED: {r.variable_name} -> {new_var}")
    else:
        new_var = r.variable_name
        unchanged_vars.append(r.variable_name)
        log.info(f"UNCHANGED: {r.variable_name}")

    updated_rows.append(
        Row(
            time=r.time,
            val=r.val,
            variable_name=new_var,
            unit=r.unit,
            network_name=r.network_name,
            station_id=r.station_id,
            lat=r.lat,
            lon=r.lon
        )
    )

# optional: summary
print(f"\nTotal updated: {len(updated_vars)}")
print(f"Total unchanged: {len(unchanged_vars)}")

INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Rain -> Rainmm
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Pressure -> Pressurembar
INFO:__main__:UPDATED: Temp -> TempC
INFO:__main__:UPDATED: Temp -> TempC
INFO:__main__:UPDATED: Temp -> TempC
INFO:__main__:UPDATED: Temp -


Total updated: 120
Total unchanged: 80


In [24]:
updated_rows

[Row(time=datetime.datetime(2021, 7, 9, 12, 0), val=nan, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 13, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 14, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 15, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 16, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 17, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barr

In [29]:
filtered_rows = []

for r in updated_rows:  # or rows if you haven't updated yet
    if r.variable_name not in exclude_vars:
        filtered_rows.append(r)
    else:
        # optional: log removal
        print(f"REMOVED: {r.variable_name}")

REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Snow depth raw
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: WC_cal
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
REMOVED: Water Content
RE

In [27]:
filtered_rows

output_folder = '/workspaces/crmprtd/rows_output/'

out_file = os.path.join(output_folder, f'updated_rows_output_Barren.pickle')
with open(out_file, 'wb') as f:
    pickle.dump(filtered_rows, f)